# Load data into landing layer

This notebook extracts French fuel price data from a CSV file and transforms it into a dimensional model (star schema) in the landing layer.

**Source**: French Government Open Data - Real-time fuel prices

**Output Tables**:
* **Dimensions**: dim_service, dim_geo, dim_station, dim_carburant
* **Facts**: fait_prix, fait_rupture

## Setup

Initialize the notebook environment by restarting Python and importing required libraries.

In [0]:
# Restart Python kernel to ensure clean state
dbutils.library.restartPython()

In [0]:
# Import standard libraries
import os
import sys

# Import PySpark functions and types for data transformation
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
# Add custom module path to Python path
base = "/Workspace/Shared/fuel-prices-lakehouse/src/carburants/transformations/"
sys.path.append(os.path.abspath(f"{base}"))

# Import custom utilities for file operations
from file_downloader import download_csv_file, read_csv_file


In [0]:
# Configuration for downloading fresh data (commented out - using existing file)
# Uncomment to download the latest data from the French Government API
# url = "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/prix-des-carburants-en-france-flux-instantane-v2/exports/csv?use_labels=true"
# path = "/Volumes/kyc/landing/data/carburants"

In [0]:
# Download the latest data file (commented out - using existing file)
# file = download_csv_file(url, path)

## Load source data

Read the fuel prices CSV file from the Unity Catalog volume.

In [0]:
# Path to the fuel prices CSV file in Unity Catalog volume
file = "/Volumes/kyc/landing/data/carburants/prix-des-carburants-en-france-flux-instantane-v2.csv"

# Read CSV file into a Spark DataFrame
carburants_df = read_csv_file(file, spark)

## Define data schemas

Define PySpark schemas for parsing nested JSON columns (services, prix, rupture).

In [0]:
# Define schemas for parsing nested JSON columns in the source data
# The CSV contains JSON-encoded strings that need explicit schemas for parsing
# Schema for 'services' column - array of service names offered at each station
schema_services = StructType([
    StructField("service", ArrayType(StringType()), True)
])

# Schema for 'prix' column - array of fuel prices with metadata
# Each price record contains: fuel name, ID, last update timestamp, and price value
schema_prix = ArrayType(StructType([
    StructField("@nom", StringType(), True),      # Fuel type name (e.g., "Gazole", "SP95")
    StructField("@id", StringType(), True),       # Fuel type ID
    StructField("@maj", StringType(), True),      # Last update timestamp
    StructField("@valeur", StringType(), True),   # Price value (€/L)
]))

# Schema for 'rupture' column - array of fuel stock-out events
# Tracks when specific fuels are unavailable at a station
schema_rupture = ArrayType(StructType([
    StructField("@nom", StringType(), True),      # Fuel type name
    StructField("@id", IntegerType(), True),      # Fuel type ID
    StructField("@debut", DateType(), True),      # Stock-out start date
    StructField("@fin", DateType(), True),        # Stock-out end date
    StructField("@type", StringType(), True),     # Type of stock-out
]))

## Transformation functions

Define functions to build dimension and fact tables from the raw data.

In [0]:
# ============================================================================
# FACT TABLE BUILDERS
# ============================================================================

def build_fait_prix(df):
    """Build fact table for fuel prices.
    Explodes the nested 'prix' JSON array to create one row per fuel type per station.

    Returns: DataFrame with columns [station_id, nom_carburant, carburant_id, date_maj, prix]
    """
    df = df.withColumn("prix_parsed", from_json(col("prix"), schema_prix))
    df = df.withColumn("carburant", explode(col("prix_parsed")))
    return df.select(
        col("id").alias("station_id"),
        col("carburant.@nom").alias("nom_carburant"),
        col("carburant.@id").cast("int").alias("carburant_id"),
        col("carburant.@maj").cast(TimestampType()).alias("date_maj"),
        col("carburant.@valeur").cast(DoubleType()).alias("prix"),
    )


def build_fait_rupture(df):
    """Build fact table for fuel stock-outs (ruptures).
     Explodes the nested 'rupture' JSON array to track unavailability periods.

    Returns: DataFrame with columns [station_id, id_carburant, nom_carburant, debut_rupture, fin_rupture, type_rupture]
    """
    rupture_parsed_df = df\
                        .withColumn("rupture_parsed",from_json(col("rupture"), schema_rupture))
    rupture_exploded_df = rupture_parsed_df\
                        .withColumn("rupture_exploded", explode(col('rupture_parsed')))
    return rupture_exploded_df.\
        select(
            col("id").alias("station_id"),
            col("rupture_exploded.@id").alias("id_carburant"),
             col("rupture_exploded.@nom").alias("nom_carburant"),
            col("rupture_exploded.@debut").alias("debut_rupture"),
            col("rupture_exploded.@fin").alias("fin_rupture"),
            col("rupture_exploded.@type").alias("type_rupture"),
            )

# ============================================================================
# DIMENSION TABLE BUILDERS
# ============================================================================

def build_dim_service(df):
    """Build dimension table for services offered at stations.
    
    Explodes the services array to create one row per service per station.
    Returns: DataFrame with columns [station_id, service]
    """
    df = df.withColumn("services_parsed", from_json(col("services"), schema_services))
    df = df.withColumn("service", explode(col("services_parsed.service")))
    return df.select(
        col("id").alias("station_id"),
        "service",
    )\
    .sort("station_id", ascending = True)

def build_dim_geo(df):
    """Build dimension table for geographic regions.
    
    Extracts unique combinations of region and department codes.
    Returns: DataFrame with columns [code_region, region, code_departement, departement]
    """
    dim_geo_df = df.select(
        "code_region",
        col("Région").alias("region"),
        "code_departement",
        col("Département").alias("departement")      
    )\
    .sort("id",ascending=True)\
    .distinct()
    return dim_geo_df

def build_dim_station(df):
    """Build dimension table for gas stations.
    
    Extracts station attributes and location information.
    Returns: DataFrame with columns [station_id, adresse, ville, code_postal, code_departement, latitude, longitude, service, code_region]
    """
    return df.select(
        col("id").alias("station_id"),
        "adresse",
        "ville",
        col("Code postal").alias("code_postal"),
        "code_departement",
        "latitude",
        "longitude",
        col("Services proposés").alias("service"),
        "code_region"
    ).sort("id",ascending=True)

def build_dim_carburant(df):
    """Build dimension table for fuel types.
    
    Extracts unique fuel types from the price data.
    Returns: DataFrame with columns [id, nom]
    """
    dim_carburant_df =  build_fait_prix(df)\
                .select(
                    col("carburant_id").alias("id"),
                     col("nom_carburant").alias("nom")
                     )\
                .sort("id",ascending=True)\
                .distinct()
    return dim_carburant_df

## Build and persist dimension tables

Create and save all dimension tables to the landing layer.

In [0]:
# ============================================================================
# BUILD DIMENSION TABLES
# ============================================================================
 
dim_service = build_dim_service(carburants_df)
dim_geo      = build_dim_geo(carburants_df)
dim_station  = build_dim_station(carburants_df)
dim_carburant = build_dim_carburant(carburants_df)

# ============================================================================
# PERSIST DIMENSION TABLES TO LANDING LAYER
# ============================================================================
# Write all dimension tables to Unity Catalog as Delta tables
# Mode: overwrite - replaces existing data on each run
dim_service.write\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable("kyc.landing.dim_service")

dim_geo.write\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable("kyc.landing.dim_geo")

dim_station.write\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable("kyc.landing.dim_station")

dim_carburant.write\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable("kyc.landing.dim_carburant")



## Build and persist fact tables

Create and save fact tables (prices and stock-outs) to the landing layer.

In [0]:
# ============================================================================
# BUILD FACT TABLES
# ============================================================================
# Drop 'nom_carburant' from facts - it's maintained in dim_carburant dimension
fait_prix    = build_fait_prix(carburants_df).drop('nom_carburant')
fait_rupture = build_fait_rupture(carburants_df).drop('nom_carburant')

# ============================================================================
# PERSIST FACT TABLES TO LANDING LAYER
# ============================================================================
# Write fact tables to Unity Catalog as Delta tables
# Mode: overwrite - replaces existing data on each run
fait_prix.write\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable("kyc.landing.fait_prix")

fait_rupture.write\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable("kyc.landing.fait_rupture")